# Chapter 20: Private Credit Analytics

This notebook accompanies the chapter in *Python & AI for Rates, Bonds, and Credit*. It focuses on reproducible inspection: run the chapter script, inspect the data snapshot, and connect the numerical result back to the fixed-income interpretation.

In [ ]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SCRIPT = ROOT / "code" / "ch20_private_credit_analytics.py"
print(SCRIPT.relative_to(ROOT))

result = subprocess.run(
    [sys.executable, str(SCRIPT)],
    cwd=ROOT,
    capture_output=True,
    text=True,
    check=True,
)
print(result.stdout)

## Interpretation checklist

Use the script output as controlled evidence. Check the units, sign convention, source dataset, and the economic direction of the result before using it in a written explanation or GenAI-assisted summary.

In [ ]:
import pandas as pd

DATA = ROOT / "data"
chapter_files = sorted(DATA.glob("ch20_*.csv"))
summary = pd.DataFrame({
    "file": [p.name for p in chapter_files],
    "rows": [pd.read_csv(p).shape[0] for p in chapter_files],
    "columns": [pd.read_csv(p).shape[1] for p in chapter_files],
})
summary

## Chapter-specific checks

Flag borrower-level covenant pressure and compare lender income with stressed borrower capacity.

In [ ]:
loans = pd.read_csv(DATA / "ch20_private_credit_loans.csv")
metrics = pd.read_csv(DATA / "ch20_borrower_metrics.csv")
book = loans.merge(metrics, on="borrower")
book["coupon_income"] = book.principal * (book.base_rate + book.spread)
book["leverage"] = book.debt / book.ebitda
book["coverage"] = book.ebitda / book.interest
book["leverage_headroom"] = book.max_leverage - book.leverage
book["review_flag"] = book.leverage_headroom.lt(0)
book[["borrower", "coupon_income", "leverage",
      "coverage", "leverage_headroom", "review_flag"]].round(3)